In [2]:
import pandas as pd
from pathlib import Path
import numpy as np

In [ ]:
#Step 1 - loading the TCGA primary tumor files
DATA = Path("../../R_scripts_dea_gsea/results/files_for_pyhton")

counts = pd.read_csv(DATA/"tcga_counts_01.tsv", sep="\t", index_col=0)
meta = pd.read_csv(DATA/"tcga_meta_01.tsv", sep="\t", index_col=0)

print("counts:", counts.shape)
print("meta:", meta.shape)

#check if the rows of metadata align with the columns of the countmatrix
print("n count cols:", counts.shape[1])
print("n meta rows:", meta.shape[0])

print("overlap:", counts.columns.isin(meta.index).sum())

counts: (63856, 304)
meta: (304, 937)
n count cols: 304
n meta rows: 304
overlap: 304


In [9]:
#Step 2 - inspect the data
#gene IDs in counts (row index)
print("Genes:")
print(counts.index[:5])
print(counts.index[-5:])

#sample IDs in counts - columns
print("Samples:")
print(counts.columns[:5])
print(counts.columns[-5:])


Genes:
Index(['ENSG00000278704.1', 'ENSG00000277400.1', 'ENSG00000274847.1',
       'ENSG00000277428.1', 'ENSG00000276256.1'],
      dtype='object')
Index(['ENSG00000124334.17_PAR_Y', 'ENSG00000185203.12_PAR_Y',
       'ENSG00000270726.6_PAR_Y', 'ENSG00000182484.15_PAR_Y',
       'ENSG00000227159.8_PAR_Y'],
      dtype='object')
Samples:
Index(['5c8c8a6e-40d4-4f82-ac3d-90cfdee15c0a',
       'd9096909-1439-462d-b69c-cad1bf4f420c',
       '29c17355-d646-48da-9e54-7b6dd85dd610',
       '29bcba51-2580-473d-9cf7-bedbd0dbad1b',
       'ad97b334-e034-4d33-a4d5-48c32d5d521c'],
      dtype='object')
Index(['6fd0cfc8-ac8d-4cf1-8c33-833ad98a30ef',
       '76084aa6-19bf-4605-b6a1-73f1d4badaf9',
       '40855647-e942-4f32-90b7-beb921929bfb',
       'b3f746dc-e487-4305-9f30-825d1520e8fd',
       '8074a7f6-2fde-4578-b71c-01c542d468fe'],
      dtype='object')


In [10]:
#meta columns -sample IDs
print("Samples in meta:")
print(meta.index[:5])

Samples in meta:
Index(['5c8c8a6e-40d4-4f82-ac3d-90cfdee15c0a',
       'd9096909-1439-462d-b69c-cad1bf4f420c',
       '29c17355-d646-48da-9e54-7b6dd85dd610',
       '29bcba51-2580-473d-9cf7-bedbd0dbad1b',
       'ad97b334-e034-4d33-a4d5-48c32d5d521c'],
      dtype='object')


In [11]:
#values in counts
print("Counts values:")
print(counts.iloc[:5, :5])

Counts values:
                   5c8c8a6e-40d4-4f82-ac3d-90cfdee15c0a  \
ENSG00000278704.1                                     0   
ENSG00000277400.1                                     0   
ENSG00000274847.1                                     0   
ENSG00000277428.1                                     0   
ENSG00000276256.1                                     0   

                   d9096909-1439-462d-b69c-cad1bf4f420c  \
ENSG00000278704.1                                     0   
ENSG00000277400.1                                     0   
ENSG00000274847.1                                     0   
ENSG00000277428.1                                     0   
ENSG00000276256.1                                     0   

                   29c17355-d646-48da-9e54-7b6dd85dd610  \
ENSG00000278704.1                                     0   
ENSG00000277400.1                                     0   
ENSG00000274847.1                                     0   
ENSG00000277428.1                      

In [13]:
#Step 3 - identify survival-related columns
survival_columns = [c for c in meta.columns if any(k in c.lower() for k in ["death", "vital", "follow"])]

print("Candidate survival columns:")
for c in survival_columns[:30]:
  print(c)

print("Total candidates:", len(survival_columns))

Candidate survival columns:
tcga.gdc_cases.demographic.year_of_death
tcga.gdc_cases.diagnoses.vital_status
tcga.gdc_cases.diagnoses.days_to_death
tcga.gdc_cases.diagnoses.days_to_last_follow_up
tcga.cgc_case_vital_status
tcga.cgc_case_days_to_death
tcga.cgc_case_follow_up
tcga.cgc_case_days_to_last_follow_up
tcga.cgc_follow_up_new_tumor_event
tcga.cgc_follow_up_new_tumor_event_after_initial_treatment
tcga.cgc_follow_up_days_to_last_follow_up
tcga.cgc_follow_up_tumor_status
tcga.cgc_follow_up_vital_status
tcga.cgc_follow_up_days_to_death
tcga.cgc_follow_up_id
tcga.xml_vital_status
tcga.xml_days_to_last_followup
tcga.xml_days_to_death
tcga.xml_has_follow_ups_information
tcga.xml_patient_death_reason
tcga.xml_source_of_patient_death_reason
Total candidates: 21


In [14]:
#Step 4 - inspecting the chosen survival columns
cols = ["tcga.gdc_cases.diagnoses.vital_status",
"tcga.gdc_cases.diagnoses.days_to_death",
"tcga.gdc_cases.diagnoses.days_to_last_follow_up"]

for c in cols:
  if c in meta.columns:
    print(f"\n{c}")
    print(meta[c].value_counts(dropna=False).head())
    print("dtype:", meta[c].dtype)


tcga.gdc_cases.diagnoses.vital_status
tcga.gdc_cases.diagnoses.vital_status
alive    233
dead      71
Name: count, dtype: int64
dtype: object

tcga.gdc_cases.diagnoses.days_to_death
tcga.gdc_cases.diagnoses.days_to_death
NaN      232
252.0      2
829.0      2
837.0      1
715.0      1
Name: count, dtype: int64
dtype: float64

tcga.gdc_cases.diagnoses.days_to_last_follow_up
tcga.gdc_cases.diagnoses.days_to_last_follow_up
NaN      58
0.0      13
27.0      3
954.0     2
540.0     2
Name: count, dtype: int64
dtype: float64


In [17]:
#Step 5 - constructing a survival table
#one row per sample, one time column, one event column
# how many rows have BOTH times missing?
mask_both_nan = meta["tcga.gdc_cases.diagnoses.days_to_death"].isna() & \
                meta["tcga.gdc_cases.diagnoses.days_to_last_follow_up"].isna()

mask_both_nan.sum()

vital = meta["tcga.gdc_cases.diagnoses.vital_status"].astype(str).str.lower()
d_death = meta["tcga.gdc_cases.diagnoses.days_to_death"]
d_fup = meta["tcga.gdc_cases.diagnoses.days_to_last_follow_up"]

event = (vital == "dead").astype(int) # 1=dead, 0=alive
time = np.where(event == 1, d_death, d_fup)
time = pd.to_numeric(time, errors="coerce")

surv = pd.DataFrame({"time_days": time, "event": event}, index=meta.index)

print(surv.shape)
print(surv["event"].value_counts())
print(surv["time_days"].isna().sum())
print(surv["time_days"].describe())



(304, 2)
event
0    233
1     71
Name: count, dtype: int64
0
count     304.000000
mean     1019.378289
std      1134.360304
min         0.000000
25%       353.750000
50%       637.000000
75%      1245.250000
max      6408.000000
Name: time_days, dtype: float64


In [18]:
#Step 6 - export the survival table
OUT = Path("../results/survival_table")
OUT.mkdir(parents=True, exist_ok=True)

surv_path = OUT/"tcga_cesc_survival.tsv"
surv.to_csv(surv_path, sep="\t")

print("Saved to:", surv_path.resolve())

Saved to: /home/annettestomakhin/Magistritöö/EMT_scoring_and_cox_model/results/survival_table/tcga_cesc_survival.tsv


In [5]:
#Step 7 - inspect the genes files
genes = pd.read_csv(DATA/"tcga_genes_rowData.tsv", sep="\t", index_col=0)

print(genes.shape)
print(genes.columns[:15])
print(genes.head(3))

(63856, 10)
Index(['source', 'type', 'bp_length', 'phase', 'gene_id', 'gene_type',
       'gene_name', 'level', 'havana_gene', 'tag'],
      dtype='object')
                    source  type  bp_length  phase            gene_id  \
ENSG00000278704.1  ENSEMBL  gene       2237    NaN  ENSG00000278704.1   
ENSG00000277400.1  ENSEMBL  gene       2179    NaN  ENSG00000277400.1   
ENSG00000274847.1  ENSEMBL  gene       1599    NaN  ENSG00000274847.1   

                        gene_type   gene_name  level havana_gene  tag  
ENSG00000278704.1  protein_coding  BX004987.1      3         NaN  NaN  
ENSG00000277400.1  protein_coding  AC145212.2      3         NaN  NaN  
ENSG00000274847.1  protein_coding  AC145212.1      3         NaN  NaN  
